In [1]:
import mbuild as mb
import gmso
from gmso.formats import interactive_networkx_atomtypes
from gmso.parameterization import apply

import warnings
warnings.filterwarnings("ignore")

/home/chris/dev/repos/gmso/gmso/core/element.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
checked_atom_types = [
    "acetophenone", #0
    "bromomethane", #1
    "cyclohexylamine", #2
    "triethylamine", #3
    "triethyl-phosphate", #4
    "2-nitropropane", #5
    "sulfolane", #6
    "benzaldehyde", #7
    "anisole", #8
    "dimethoxymethane", #9
    "123-propanetriol", #10
    "tert-butylamine", #11
    "1-chloronaphthalene", #12
    "2-iodopropane", #13
    "2-methylpyridine", #14
    "4-methylpyridine", #15
    "diphenyl-ether", #16
    "benzyl-alcohol", #17
    "pyrrolidine", #18
    "methyl-salicylate", #19
    "1234-tetrafluorobenzene", #20
    "benzenethiol", #21
    "2-chloroaniline", #22
    "ethyl-vinyl-ether", #23
    "3-methylpyridine", #24
    "11-dichloroethene", #25
    "trifluoromethyl-benzene", #26
    "diisopropylamine", #27
    #"furan",
    #"quinoline",
]

In [44]:
index = 3
mol = mb.load(f"testing_files/{checked_atom_types[index]}.mol2")
print(f"{checked_atom_types[index]}")
top = mol.to_gmso()
top.identify_connections()
opls = gmso.core.forcefield.ForceField(xml_loc="oplsaa.xml")
apply(top=top, forcefields=opls, ignore_params=["bond", "angle", "proper", "improper", "dihedral"])

triethylamine


<Topology Topology, 22 sites,
 105 connections,
 127 potentials,
 id: 140164880819520>

In [40]:
#for site in top.sites:
#    print(site.atom_type.name, site.element.symbol)

In [45]:
mol.visualize(backend="py3Dmol")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
for mol_name in checked_atom_types:
    print(mol_name)
    mol = mb.load(f"testing_files/{mol_name}.mol2")
    top = mol.to_gmso()
    top.identify_connections()
    opls = gmso.core.forcefield.ForceField(xml_loc="oplsaa.xml")
    apply(top=top, forcefields=opls, ignore_params=[
        #"bond",
        #"angle",
        "proper",
        "improper",
        "dihedral"]
    )

acetophenone
bromomethane
cyclohexylamine
triethylamine
triethyl-phosphate
2-nitropropane
sulfolane
benzaldehyde
anisole
dimethoxymethane
123-propanetriol
tert-butylamine
1-chloronaphthalene
2-iodopropane
2-methylpyridine
4-methylpyridine
diphenyl-ether
benzyl-alcohol
pyrrolidine
methyl-salicylate
1234-tetrafluorobenzene
benzenethiol
2-chloroaniline
ethyl-vinyl-ether
3-methylpyridine
11-dichloroethene
trifluoromethyl-benzene
diisopropylamine


# Check missing dihedrals:

In [18]:
missing_dihedrals = set()
missing_impropers = set()

for mol_name in checked_atom_types:
    print(mol_name)
    mol = mb.load(f"testing_files/{mol_name}.mol2")
    top = mol.to_gmso()
    top.identify_connections()
    opls = gmso.core.forcefield.ForceField(xml_loc="oplsaa.xml")
    apply(
        top=top,
        forcefields=opls,
        ignore_params=[
            #"bond",
            #"angle",
            "proper",
            "improper",
            "dihedral"
        ],
        remove_untyped=False
    )

    for dih in top.dihedrals:
        if dih.dihedral_type is None:
            conn_types = tuple([site.atom_type.atomclass for site in dih.connection_members])
            missing_dihedrals.add(conn_types)

    for imp in top.impropers:
        if imp.improper_type is None:
            conn_types = tuple([site.atom_type.atomclass for site in imp.connection_members])
            missing_impropers.add(conn_types)

acetophenone
bromomethane
cyclohexylamine
triethylamine
triethyl-phosphate
2-nitropropane
sulfolane
benzaldehyde
anisole
dimethoxymethane
123-propanetriol
tert-butylamine
1-chloronaphthalene
2-iodopropane
2-methylpyridine
4-methylpyridine
diphenyl-ether
benzyl-alcohol
pyrrolidine
methyl-salicylate
1234-tetrafluorobenzene
benzenethiol
2-chloroaniline
ethyl-vinyl-ether
3-methylpyridine
11-dichloroethene
trifluoromethyl-benzene
diisopropylamine


In [19]:
for dih in missing_dihedrals:
    print(dih)

### Found dihedrals:
Line 2702
CT     OS     C     O_2

Line 2359
CA     CA     C_2    O_2

Line 2481
CM     OS     CT     CT

Line 2359
CA     CA     C_2     O_2


Line 2865
HC     CT     CA     NC

Line 2793
I     CT     CT     HC

Line 2929
CA     C_2     CT     HC

In [17]:
len(missing_impropers)

85

In [16]:
for imp in missing_impropers:
    print(imp)

('CT', 'HC', 'HC', 'C_2')
('CT', 'HC', 'HC', 'CA')
('CA', 'C_2', 'CA', 'CA')
('C_2', 'CT', 'O_2', 'CA')
('CA', 'CA', 'HA', 'NC')
('CT', 'CT', 'HC', 'NO')
('CT', 'CT', 'HC', 'OH')
('NT', 'CA', 'H', 'H')
('CM', 'CM', 'CL', 'CL')
('CM', 'CM', 'HC', 'HC')
('CO', 'OS', 'HC', 'OS')
('CM', 'OS', 'HC', 'CM')
('NT', 'CT', 'H', 'H')
('CA', 'CA', 'CA', 'OS')
('C', 'OS', 'O_2', 'CA')
('CT', 'OH', 'HC', 'HC')
('CT', 'HC', 'HC', 'OS')
('CT', 'CT', 'HC', 'NT')
('CT', 'OH', 'HC', 'CT')
('CA', 'F', 'CA', 'CA')
('P', 'OS', 'O2', 'OS')
('CT', 'CT', 'CT', 'I')
('CT', 'CA', 'HC', 'OH')
('CA', 'CT', 'CA', 'NC')
('CT', 'CT', 'HC', 'HC')
('CT', 'NT', 'HC', 'HC')
('CA', 'CA', 'NC', 'HA')
('CT', 'CT', 'CT', 'NO')
('CT', 'CT', 'OH', 'CT')
('CT', 'CT', 'HC', 'CT')
('CA', 'CA', 'F', 'CA')
('CT', 'HC', 'HC', 'Br')
('CT', 'NT', 'HC', 'CT')
('CT', 'CA', 'F', 'F')
('CA', 'CT', 'CA', 'CA')
('CA', 'CA', 'CA', 'HA')
('P', 'OS', 'OS', 'O2')
('NT', 'CT', 'H', 'CT')
('CT', 'F', 'F', 'F')
('CA', 'NC', 'HA', 'CA')
('CA', 'CA'

In [5]:
index = 0
print(checked_atom_types[index])
mol = mb.load(f"testing_files/{checked_atom_types[index]}.mol2")
top = mol.to_gmso()
top.identify_connections()
opls = gmso.core.forcefield.ForceField(xml_loc="oplsaa.xml")
apply(
    top=top,
    forcefields=opls,
    remove_untyped=False,
    ignore_params=[
        #"bond",
        #"angle",
        "improper",
        "dihedral"
    ]
)

missing_dihedrals = set()
for dih in top.dihedrals:
    if dih.dihedral_type is None:
        conn_types = tuple([site.atom_type.atomclass for site in dih.connection_members])
        missing_dihedrals.add(conn_types)

print("Missing dihedrals")
print(missing_dihedrals)

acetophenone
Missing dihedrals
{('HC', 'CT', 'C_2', 'CA'), ('O_2', 'C_2', 'CA', 'CA')}


In [23]:
missing_dihedrals

{('HC', 'CT', 'C_2', 'CA'), ('O_2', 'C_2', 'CA', 'CA')}

In [ ]:
for site in top.sites:
    print(site.atom_type.name)